In [2]:
import torch
from torch import nn
import torchvision
from torchvision import transforms


In [3]:
class MLP(nn.Module):
    def __init__(self, hidden_dim, output_dim, lr=1e-1):
        super().__init__()
        self.lr = lr
        self.Layer1 = nn.LazyLinear(hidden_dim)
        self.Layer2 = nn.LazyLinear(output_dim)
        self.net = nn.Sequential(nn.Flatten(), self.Layer1, nn.ReLU(), self.Layer2)
        self.loss_function = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.net(X)

    def loss(self, y_hat, y):
        return self.loss_function(y_hat, y)

    def get_optimizer(self):
        return torch.optim.SGD(self.parameters(), lr=self.lr, weight_decay=5e-3)

In [4]:
class DropoutMLP(nn.Module):
    def __init__(self, hidden_dim_1, hidden_dim_2, output_dim, dropout_1, dropout_2, lr=1e-1):
        super().__init__()
        self.lr = lr
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(hidden_dim_1), nn.ReLU(), nn.Dropout(dropout_1), nn.LazyLinear(hidden_dim_2),
                                 nn.ReLU(), nn.Dropout(dropout_2), nn.LazyLinear(output_dim))
        self.loss_function = nn.CrossEntropyLoss()

    def forward(self, X):
        return self.net(X)

    def loss(self, y_hat, y):
        return self.loss_function(y_hat, y)

    def get_optimizer(self):
        return torch.optim.SGD(self.parameters(), lr=self.lr, weight_decay=5e-3)

In [5]:
class FashionMNIST():
    def __init__(self, batch_size=64, resize=(28, 28)):
        self.batch_size = batch_size
        self.resize = resize
        self.transform = transforms.Compose([transforms.Resize(self.resize),
                                             transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(root='./root', train=True, download=True,
                                                       transform=self.transform)
        self.validation = torchvision.datasets.FashionMNIST(root='./root', train=False, download=True,
                                                            transform=self.transform)

    def get_dataloader(self, train):
        data = self.train if train else self.validation
        dataloader = torch.utils.data.DataLoader(data, batch_size=self.batch_size, shuffle=train)
        return dataloader

    def training_data(self):
        return self.get_dataloader(True)

    def validation_data(self):
        return self.get_dataloader(False)

In [6]:
class Dataloader():
    def __init__(self, batch_size):
        self.batch_size = batch_size

        self.train_data = torchvision.datasets.FashionMNIST(root='/home/beret/Documents/moje_projekty/pytorch_study/4_Linear_classification/data', train=True, download=True, transform=transforms.ToTensor())

    def training_data(self):
        pass

    def validation_data(self):
        pass

    def get_dataloader(self, train=True):
        return
    def get_dataloader(self, train=True):

In [7]:
def Train_batches(dataloader, model, optimizer):

    model.train()
    for (X, y) in dataloader:
        pred = model(X)
        loss = model.loss(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


def validate_batches(dataloader, model):

    total_loss, correct = 0, 0
    model.eval()
    for (X, y) in dataloader:
        with torch.no_grad():
            y_hat = model(X)
            total_loss += model.loss(y_hat, y).item()
            correct += (y_hat.argmax(dim=1) == y).float().sum().item()

    total_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"validation loss: {total_loss :> 8f} and correct: {(100 * correct) :>0.1f}%")

In [8]:
data = FashionMNIST(batch_size=256)
data_train = data.training_data()
data_validation = data.validation_data()
model = MLP(hidden_dim=512,output_dim=10, lr=1e-1)
optim = model.get_optimizer()

for i in range(10):
    Train_batches(data_train, model, optim)
    validate_batches(data_validation, model)

validation loss:  0.617090 and correct: 78.7%
validation loss:  0.534722 and correct: 81.7%
validation loss:  0.503512 and correct: 82.8%
validation loss:  0.496674 and correct: 82.6%
validation loss:  0.516246 and correct: 81.0%
validation loss:  0.503828 and correct: 82.5%
validation loss:  0.493807 and correct: 82.7%
validation loss:  0.480989 and correct: 83.1%
validation loss:  0.467758 and correct: 83.8%
validation loss:  0.482717 and correct: 82.7%


In [9]:
data = FashionMNIST(batch_size=256)
data_train = data.training_data()
data_validation = data.validation_data()
model = DropoutMLP(hidden_dim_1=256, hidden_dim_2=256, dropout_1=0.5, dropout_2=0.5, output_dim=10, lr=1e-1)
optim = model.get_optimizer()

for i in range(10):
    Train_batches(data_train, model, optim)
    validate_batches(data_validation, model)

validation loss:  0.742666 and correct: 70.3%
validation loss:  0.614720 and correct: 76.8%
validation loss:  0.538803 and correct: 80.5%
validation loss:  0.504184 and correct: 82.0%
validation loss:  0.489035 and correct: 81.9%
validation loss:  0.506716 and correct: 81.2%
validation loss:  0.480977 and correct: 82.6%
validation loss:  0.474976 and correct: 82.6%
validation loss:  0.453743 and correct: 83.5%
validation loss:  0.466156 and correct: 83.0%
